In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Класичний зворотний зв'язок і керування потоком (динамічні схеми)

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



# Класичний зворотний зв'язок і керування потоком


<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці розроблено з використанням наведених нижче вимог.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
> **Note:** Нова версія динамічних схем тепер доступна всім користувачам на всіх бекендах. Ти можеш запускати динамічні схеми у промисловому масштабі. Дивись [оголошення](/announcements/product-updates/2025-09-25-new-dynamic-circuits) для отримання додаткових деталей.

Динамічні схеми — потужний інструмент, який дозволяє вимірювати кубіти посередині виконання квантової схеми і на основі результатів цих проміжних вимірювань виконувати класичні логічні операції всередині самої схеми. Цей процес також відомий як _класичний зворотний зв'язок_ (classical feedforward). Хоча ми ще на початку шляху розуміння того, як найкраще використовувати динамічні схеми, квантова дослідницька спільнота вже визначила низку сценаріїв застосування, зокрема:

* Ефективна підготовка квантових станів, наприклад [GHZ-стан,](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) [W-стан,](https://arxiv.org/abs/2403.07604) (докладніше про W-стан — у статті ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), а також широкий клас [матричних добутків станів](https://arxiv.org/abs/2404.16083)
* [Ефективне заплутування на великі відстані](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) між кубітами на одному чипі завдяки неглибоким схемам
* Ефективна [вибірка з IQP-подібних схем](https://arxiv.org/pdf/2505.04705)

Переваги, які надають динамічні схеми, водночас мають свої компроміси. Проміжні вимірювання та класичні операції зазвичай потребують більше часу на виконання, ніж двокубітні вентилі, і це збільшення часу може зводити нанівець вигоди від меншої глибини схеми. Тому скорочення тривалості проміжних вимірювань є ключовим напрямком покращень у новій [версії](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) динамічних схем, яку випускає IBM Quantum&reg;.

[Специфікація OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) визначає низку структур керування потоком, але Qiskit Runtime наразі підтримує лише умовний оператор `if`. У Qiskit SDK йому відповідає метод [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) класу [QuantumCircuit.](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit) Цей метод повертає [контекстний менеджер](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) і зазвичай використовується в операторі `with`. У цьому посібнику описано, як застосовувати цей умовний оператор.

> **Note:** Приклади коду в цьому посібнику використовують стандартну інструкцію вимірювання для проміжних вимірювань. Однак рекомендується натомість використовувати інструкцію [`MidCircuitMeasure`](/guides/measure-qubits#midcircuit), якщо бекенд її підтримує. Дивись [документацію з проміжних вимірювань](/guides/measure-qubits#mid-circuit-measurements) для отримання деталей.
## Оператор `if`
Оператор `if` використовується для умовного виконання операцій залежно від значення класичного біта або регістра.

У наведеному нижче прикладі застосовується вентиль Адамара до кубіта і виконується його вимірювання. Якщо результат дорівнює 1, то до кубіта застосовується вентиль X, який перевертає його назад у стан 0. Потім кубіт вимірюється знову. Результат повторного вимірювання має дорівнювати 0 з імовірністю 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

Оператору `with` можна передати ціль присвоєння, яка сама є контекстним менеджером — її можна зберегти та згодом використати для створення блоку `else`, що виконується тоді, коли вміст блоку `if` *не* виконується.

У наведеному нижче прикладі ініціалізуються регістри з двома кубітами та двома класичними бітами. До першого кубіта застосовується вентиль Адамара і виконується його вимірювання. Якщо результат дорівнює 1, до другого кубіта застосовується вентиль Адамара; інакше — вентиль X. Наприкінці вимірюється й другий кубіт.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Крім умови на один класичний біт, можна також задавати умову на значення класичного регістра, що складається з кількох бітів.

У наведеному нижче прикладі до двох кубітів застосовуються вентилі Адамара і виконується їх вимірювання. Якщо результат дорівнює `01` — тобто перший кубіт дорівнює 1, а другий — 0 — до третього кубіта застосовується вентиль X. Наприкінці вимірюється третій кубіт. Зауваж, що для наочності ми явно вказали стан третього класичного біта (0) в умові `if`. На схематичному зображенні схеми умова позначається кружечками на класичних бітах, на яких задається умова. Чорний кружечок означає умову на 1, а білий — умову на 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Класичні вирази
Модуль класичних виразів Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) містить експериментальне представлення операцій виконання над класичними значеннями під час виконання схеми. Через апаратні обмеження наразі підтримуються лише умови `QuantumCircuit.if_test()`.

Наступний приклад демонструє, що за допомогою обчислення парності можна створити n-кубітний GHZ-стан, використовуючи динамічні схеми. Спочатку генеруються $n/2$ пар Белла на сусідніх кубітах. Потім ці пари з'єднуються між собою шаром вентилів CNOT між парами. Далі вимірюється цільовий кубіт усіх попередніх вентилів CNOT, і кожен виміряний кубіт скидається до стану $\vert 0 \rangle$. До кожного невиміряного вузла, для якого парність усіх попередніх бітів непарна, застосовується $X$. Нарешті, до виміряних кубітів застосовуються вентилі CNOT для відновлення заплутаності, втраченої під час вимірювання.

При обчисленні парності перший елемент побудованого виразу передбачає підняття (lift) об'єкта Python `mr[0]` до вузла [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` використовується для перетворення довільних об'єктів на класичні вирази). Для `mr[1]` та можливих наступних класичних регістрів це не потрібно, оскільки вони є вхідними аргументами `expr.bit_xor`, а необхідне підняття виконується автоматично. Такі вирази можна будувати в циклах та інших конструкціях.

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

## Пошук бекендів із підтримкою динамічних схем
Щоб знайти всі бекенди, доступні твоєму акаунту і такі, що підтримують динамічні схеми, виконай код на зразок наведеного нижче. У цьому прикладі передбачається, що ти [зберіг свої облікові дані.](/guides/save-credentials) Також можна [явно вказати облікові дані](/guides/initialize-account#explicit) під час ініціалізації облікового запису служби Qiskit Runtime. Це дозволяє, наприклад, переглянути бекенди, доступні для конкретного екземпляра або типу тарифного плану.

> **Note:** - Бекенди, доступні акаунту, залежать від екземпляра, вказаного в облікових даних.
> - Нова версія динамічних схем тепер доступна всім користувачам на всіх бекендах. Дивись [оголошення](/announcements/product-updates/2025-09-25-new-dynamic-circuits) для отримання додаткових деталей.